# Build E5 Index on GPU
Mount Google Drive, read `chunks.json`, encode with multilingual-e5-base, save the FAISS index back to Drive.

In [7]:
!pip install -q sentence-transformers faiss-cpu

### Step 1: Mount Google Drive
Run this cell, then follow the authorization link.
Before running the next cell, upload `data/processed/chunks.json` to **My Drive** (the root of your Google Drive).

In [8]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


### Step 2: Encode and build index
Make sure `chunks.json` is in My Drive root before running.

In [9]:
import json
import time

import faiss
import numpy as np
from sentence_transformers import SentenceTransformer

chunks = json.loads(open("/content/drive/MyDrive/chunks.json", "r", encoding="utf-8").read())
print(f"Loaded {len(chunks)} chunks")

model = SentenceTransformer("intfloat/multilingual-e5-base")
texts = [f"passage: {c['content']}" for c in chunks]

t0 = time.time()
embeddings = model.encode(
    texts,
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True,
)
elapsed = time.time() - t0
print(f"Encoding done: {elapsed:.1f}s ({len(texts)/elapsed:.1f} chunks/s)")

dim = embeddings.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(embeddings.astype(np.float32))
print(f"FAISS index: {index.ntotal} vectors, dim={dim}")

Loaded 3141 chunks


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/50 [00:00<?, ?it/s]

Encoding done: 36.6s (85.7 chunks/s)
FAISS index: 3141 vectors, dim=768


### Step 3: Save results to Drive
The 3 files will appear in My Drive — download them from there.

In [ ]:
out = "/content/drive/MyDrive"

faiss.write_index(index, f"{out}/faiss_e5.index")
np.save(f"{out}/embeddings.npy", embeddings)

ids = [c["chunk_id"] for c in chunks]
with open(f"{out}/faiss_ids.json", "w", encoding="utf-8") as f:
    json.dump(ids, f, ensure_ascii=False)

print("Saved to Google Drive:")
print("  - faiss_e5.index")
print("  - embeddings.npy")
print("  - faiss_ids.json")
print("\nDownload these 3 files from your Drive, then copy them to data/search/")

Saved to Google Drive:
  - faiss_e5.index
  - embeddings.npy
  - faiss_ids.json

Download these 3 files from your Drive, then copy them to data/search/


: 

: 